# Smart Civic Grievance Portal — Gujarati Handwritten Word OCR

**Beginner Kaggle notebook · PyTorch CRNN + CTC · Unicode Gujarati output**

**Purpose:** Train a recognizer for images of **single handwritten Gujarati words**. The supplied Hugging Face dataset is word-level; training on it **does not** make a complete handwritten page recognizable. After training, run the model on cropped words or add a separately validated page/line/word segmenter. Use an LLM *after OCR* for department classification.

1. Go to https://www.kaggle.com/code and create **New Notebook**.
2. In notebook **Settings**, enable **Internet**, and select a **GPU accelerator** (if available).
3. Upload/import this notebook, then click **Run All**.
4. The final file `gujarati_crnn.pt` appears in `/kaggle/working/` and can be downloaded under **Output**.
5. Copy that file beside `gujarati_ocr.py` and `app_trained_ocr.py` on your computer.

**Experiment first:** the default limits below train on only a fraction of the full data and will often have poor accuracy. Raise them and train longer only after this notebook runs correctly. No model weights or accuracy claims are supplied in advance.

## Cell 1 — Install Hugging Face dataset loader
Kaggle already includes PyTorch in normal GPU images. We install the dataset package; **do not** reinstall CUDA/PyTorch unless you have a specific compatibility issue.

In [ ]:
%pip -q install "datasets>=3,<5" pillow tqdm

## Cell 2 — Import libraries and see if a GPU is available
`cuda` means the NVIDIA GPU will run the CRNN. If it says `cpu`, go back to Kaggle Settings and enable a GPU.

In [ ]:
import random, time, unicodedata
from itertools import islice
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from PIL import Image, ImageOps
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Cell 3 — Choose dataset sizes
`6,000` training samples is intended to check that the pipeline works, **not** to achieve high handwriting accuracy. The complete hosted dataset is about **4 GB**; larger settings require more time, bandwidth and data. Streaming avoids downloading every Parquet shard immediately.

In [ ]:
DATASET_ID = "prthm29/Gujarati-Handwritten-Dataset"
TRAIN_LIMIT = 6000     # Begin with 6000; later try 20000 and then full data
VAL_LIMIT = 600
TEST_LIMIT = 600
EPOCHS = 12           # Increase after an initial successful run
BATCH_SIZE = 32       # Use 16 if out of GPU memory
LEARNING_RATE = 1e-3
IMG_H, IMG_W = 32, 256

## Cell 4 — Load real image/text pairs from Hugging Face
The current hosted version supplies `image` (a PIL image), `text` (already Gujarati), and `file_name`. No `vocab.txt` translation is necessary for **this Parquet version**. It exposes `train`, `val`/`validation`, and `test` splits. Check the real keys; metadata counts in the dataset card are internally inconsistent.

In [ ]:
dataset = load_dataset(DATASET_ID, streaming=True)
print("Available splits:", list(dataset.keys()))
# Optional: ask the Hugging Face Dataset Server for the current exact row counts.
try:
    import requests
    response = requests.get("https://datasets-server.huggingface.co/splits",
                            params={"dataset": DATASET_ID}, timeout=20)
    response.raise_for_status()
    split_counts = [(x["split"], x["num_rows"]) for x in response.json()["splits"]]
    print("Current hosted split counts:", split_counts)
except Exception as error:
    print("Could not fetch remote split counts; this does NOT block training:", error)
train_key = "train"
val_key = "validation" if "validation" in dataset else "val"
test_key = "test"
first = next(iter(dataset[train_key]))
print("Columns:", list(first.keys()))
print("Example Gujarati label:", first["text"])
display(first["image"].resize((400, 90)))

## Cell 5 — Resize each word and create a small dataset in RAM
We turn each word into a grayscale 32 × 256 image (preserving its aspect ratio, white padding on the right). This must exactly match the preprocessing in the Streamlit inference script. We store compact 8-bit arrays so the notebook doesn't keep thousands of large decoded PIL images in memory.

In [ ]:
def prepare_word_image(image, height=IMG_H, width=IMG_W):
    image = ImageOps.exif_transpose(image).convert("L")
    new_w = max(1, min(width, round(image.width * height / image.height)))
    image = image.resize((new_w, height), Image.Resampling.LANCZOS)
    canvas = Image.new("L", (width, height), 255)
    canvas.paste(image, (0, 0))
    return np.asarray(canvas, dtype=np.uint8).copy()


def take_examples(iterable_split, count):
    samples = []
    for row in tqdm(islice(iterable_split, count), total=count):
        label = unicodedata.normalize("NFC", str(row["text"]).strip())
        if not label:
            continue
        try:
            pixels = prepare_word_image(row["image"])
            samples.append({"pixels": pixels, "label": label})
        except (ValueError, OSError) as error:
            print("Skipped unreadable image:", error)
    return samples

train_examples = take_examples(dataset[train_key], TRAIN_LIMIT)
val_examples = take_examples(dataset[val_key], VAL_LIMIT)
test_examples = take_examples(dataset[test_key], TEST_LIMIT)
print("Loaded:", len(train_examples), len(val_examples), len(test_examples))
assert len(train_examples) > 0 and len(val_examples) > 0 and len(test_examples) > 0
print("First label:", train_examples[0]["label"])
display(Image.fromarray(train_examples[0]["pixels"]).resize((768, 96)))

## Cell 6 — Build the Gujarati character vocabulary
A model doesn't understand Unicode letters directly: we map each Unicode code point to a number. **0** is reserved for the CTC `blank` symbol. Gujarati vowels and combining marks count as separate code points here.

The vocabulary is learned **only from training labels**. Any validation/test sample containing a previously unseen code point must be excluded, and we print how many are excluded. This is a limitation of small-subset experiments, not a claim of accuracy on all words.

In [ ]:
chars = sorted(set("".join(item["label"] for item in train_examples)))
char_to_id = {ch: i + 1 for i, ch in enumerate(chars)}
print("Characters:", len(chars), "| classes including CTC blank:", len(chars) + 1)
print("Sample character IDs:", list(char_to_id.items())[:20])

# CTC needs extra alignment space for repeated consecutive symbols.
def ctc_required_steps(word):
    return len(word) + sum(word[i] == word[i - 1] for i in range(1, len(word)))

def compatible(samples):
    return [s for s in samples if set(s["label"]).issubset(char_to_id)
            and ctc_required_steps(s["label"]) <= IMG_W // 4]

train_ok = compatible(train_examples)
val_ok = compatible(val_examples)
test_ok = compatible(test_examples)
print("Retained train/val/test:", len(train_ok), len(val_ok), len(test_ok))
print("Excluded val/test:", len(val_examples)-len(val_ok), len(test_examples)-len(test_ok))
assert len(train_ok) > 0 and len(val_ok) > 0 and len(test_ok) > 0

## Cell 7 — PyTorch Dataset and mini-batches
Image pixel data are normalized to a number from 0 to 1. Labels are converted to lists of integer character IDs. Unlike regular text classification, CTC labels are concatenated into one long tensor with a length for each word.

In [ ]:
class WordDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        item = self.samples[index]
        image = torch.from_numpy(item["pixels"].copy()).unsqueeze(0).float() / 255.0
        target = [char_to_id[ch] for ch in item["label"]]
        return image, torch.tensor(target, dtype=torch.long), item["label"]

def collate_words(batch):
    images, targets, labels = zip(*batch)
    return (torch.stack(images), torch.cat(targets),
            torch.tensor([len(t) for t in targets], dtype=torch.long), list(labels))

train_loader = DataLoader(WordDataset(train_ok), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_words, num_workers=0)
val_loader = DataLoader(WordDataset(val_ok), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_words, num_workers=0)
test_loader = DataLoader(WordDataset(test_ok), batch_size=BATCH_SIZE,
                         shuffle=False, collate_fn=collate_words, num_workers=0)
print("Training batches:", len(train_loader))

## Cell 8 — Define CRNN model (CNN + BiLSTM + CTC)
The CNN extracts handwriting strokes, the bidirectional LSTM follows them from left-to-right and right-to-left, and a dense head predicts character probabilities at each of 64 time positions. The loss in the next cell is **CTC**. This model is intentionally modest enough to experiment with on Kaggle.

In [ ]:
class GujaratiCRNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, None)),
        )
        self.rnn = nn.LSTM(256, 128, num_layers=2, bidirectional=True,
                           batch_first=True, dropout=0.1)
        self.head = nn.Linear(256, n_classes)
    def forward(self, x):
        features = self.features(x).squeeze(2).permute(0, 2, 1)
        features, _ = self.rnn(features)
        return self.head(features)

model = GujaratiCRNN(n_classes=len(chars) + 1).to(DEVICE)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
with torch.no_grad():
    dummy = model(torch.zeros(2, 1, IMG_H, IMG_W, device=DEVICE))
print("Model output (B, time positions, character classes):", tuple(dummy.shape))
assert dummy.shape[1] == IMG_W // 4

## Cell 9 — CTC decoding and accuracy metric (CER)
The network emits many predictions; greedy CTC decoding removes repeated consecutive symbols and blanks. `CER` = character error rate: lower is better, **0.0 is perfect**. Word accuracy is stricter: a word with even one incorrect character counts as wrong. Both numbers are measured on unseen examples.

In [ ]:
def decode_ctc(logits, chars):
    result = []
    for sequence in logits.argmax(-1).detach().cpu().tolist():
        previous, letters = 0, []
        for token in sequence:
            if token != 0 and token != previous:
                letters.append(chars[token - 1])
            previous = token
        result.append("".join(letters))
    return result


def levenshtein(a, b):
    previous = list(range(len(b) + 1))
    for i, ac in enumerate(a, 1):
        current = [i]
        for j, bc in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1,
                               previous[j - 1] + (ac != bc)))
        previous = current
    return previous[-1]

@torch.inference_mode()
def evaluate(loader):
    model.eval()
    edits, total_chars, correct, total_words = 0, 0, 0, 0
    samples = []
    for images, targets, target_lengths, references in loader:
        logits = model(images.to(DEVICE))
        predictions = decode_ctc(logits, chars)
        for predicted, reference in zip(predictions, references):
            edits += levenshtein(predicted, reference)
            total_chars += len(reference)
            correct += int(predicted == reference)
            total_words += 1
            if len(samples) < 5:
                samples.append((reference, predicted))
    return {"CER": edits / max(1, total_chars),
            "Word accuracy": correct / max(1, total_words),
            "examples": samples}

## Cell 10 — Train the model
Each epoch shows your model another pass through the training samples. Save the best checkpoint based on validation CER. **First runs may output empty or nonsense text until the model learns to align**. If training or validation CER does not improve, debug labels and preprocessing before increasing the epochs.

In [ ]:
BEST_PATH = "/kaggle/working/gujarati_crnn.pt"
# For non-Kaggle execution, change BEST_PATH to "gujarati_crnn.pt".
import os
os.makedirs(os.path.dirname(BEST_PATH), exist_ok=True)
best_cer = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    start = time.perf_counter()
    for images, targets, target_lengths, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)  # B,T,C
        log_probs = logits.log_softmax(2).permute(1, 0, 2).contiguous()  # T,B,C
        input_lengths = torch.full((images.size(0),), logits.size(1), dtype=torch.long)
        # Keeping CTCLoss lengths on CPU works with standard PyTorch CUDA backends.
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite loss: inspect image dimensions and labels")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()

    val_results = evaluate(val_loader)
    print(f"Epoch {epoch}: train loss={total_loss / max(1, len(train_loader)):.4f}, "
          f"val CER={val_results['CER']:.4f}, "
          f"val word accuracy={val_results['Word accuracy']:.4f}, "
          f"time={time.perf_counter()-start:.1f}s")
    print("Example predictions:", val_results["examples"][:3])
    if val_results["CER"] < best_cer:
        best_cer = val_results["CER"]
        torch.save({"model_state": model.state_dict(), "chars": chars,
                    "image_height": IMG_H, "image_width": IMG_W}, BEST_PATH)
        print("Saved improved checkpoint:", BEST_PATH)
print("Training complete. Best validation CER:", round(best_cer, 4))

## Cell 11 — Load your best model and evaluate the TEST set
Use the test split only after model selection, not to choose training settings. CER is measured at Unicode-code-point level; grapheme-based quality can differ. The default small subset will not be representative of every handwriting style.

In [ ]:
best = torch.load(BEST_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(best["model_state"])
model.to(DEVICE)
final_test = evaluate(test_loader)
print("TEST CER:", final_test["CER"])
print("TEST word accuracy:", final_test["Word accuracy"])
for reference, predicted in final_test["examples"]:
    print("Expected:", reference, "| OCR:", predicted)

## Cell 12 — Test one image and benchmark inference time
This measures **one cropped word**, not an entire page and not an Ollama API call. GPU warmup and transfer matter. The first request also includes model initialization in a real application.

In [ ]:
example = test_ok[0]
display(Image.fromarray(example["pixels"]).resize((768, 96)))
model.eval()
word = torch.from_numpy(example["pixels"].copy()).unsqueeze(0).unsqueeze(0).float().div(255).to(DEVICE)
with torch.inference_mode():
    for _ in range(3):
        _ = model(word)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    start = time.perf_counter()
    logits = model(word)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
elapsed_ms = (time.perf_counter() - start) * 1000
print("Correct:", example["label"])
print("Predicted:", decode_ctc(logits, chars)[0])
print("Neural network forward-pass time (one word):", round(elapsed_ms, 2), "ms")
print("A full letter requires multiple words, segmentation and an LLM, so end-to-end time is longer.")

## Cell 13 — Download the trained model from Kaggle
Click **Save Version** if you want notebook outputs persisted. The file is named `gujarati_crnn.pt` and includes model weights **plus** the exact Gujarati vocabulary. Copy the file next to the supplied local Python scripts. Do not try to copy the `.ipynb` into your Streamlit app — the exported `.pt` file is the trained model.

In [ ]:
import os
print("Your model file:", BEST_PATH)
print("Exists:", os.path.isfile(BEST_PATH))
print("Size (MB):", round(os.path.getsize(BEST_PATH) / 1024 / 1024, 2))

## What next?
1. Put `gujarati_crnn.pt`, `gujarati_ocr.py`, and `app_trained_ocr.py` together in one folder on your computer.
2. Install `streamlit ollama torch pillow numpy opencv-python-headless` in your local Python environment.
3. In a terminal run `ollama pull llama3.2` and `ollama pull llava` (assuming Ollama is installed and running).
4. Run `streamlit run app_trained_ocr.py`.
5. Test **a cropped Gujarati word** first, then cautiously test a full page using the **experimental** segmentation tab and correct the transcript before routing.
6. For an accurate full-letter Gujarati system, label many real complaint **lines/pages**, assess a pretrained Gujarati-capable document model, and fine-tune/evaluate with that data.
7. Train larger samples, inspect poor predictions, and re-run tests across different writers, photo lighting, and smartphone cameras.

**No accuracy or millisecond promise:** a modest word OCR model may have low single-word inference latency on the right GPU, but full-page extraction plus LLM typically takes substantially longer. The notebook measures it rather than fabricating results.